<a href="https://colab.research.google.com/github/Nedja995/ds-clinic/blob/main/examples/Market_a_Jet_Backpack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 Google LLC.

In [5]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Create a marketing campaign from a product sketch of a Jet Backpack

This notebook contains a code example of using the Gemini API to analyze a a product sketch (in this case, a drawing of a Jet Backpack), create a marketing campaign for it, and output taglines in JSON format.

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Market_a_Jet_Backpack.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

## Setup

In [2]:
%pip install -U -q "google-genai>=1.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.8/728.8 kB 15.0 MB/s eta 0:00:00


In [3]:
import PIL.Image
from IPython.display import display, Image, HTML
import ipywidgets as widgets

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see the [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) quickstart for an example.

In [4]:
from google import genai
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

SecretNotFoundError: Secret GOOGLE_API_KEY does not exist.

Additionally, select the model you want to use from the available options below:

In [ ]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

## Marketing Campaign
- Product Name
- Description
- Feature List / Descriptions
- H1
- H2


## Analyze Product Sketch

First you will download a sample image to be used:

In [ ]:
productSketchUrl = "https://storage.googleapis.com/generativeai-downloads/images/jetpack.jpg"
!curl -o jetpack.jpg {productSketchUrl}

You can view the sample image to understand the prompts you are going to work with:

In [ ]:
img = PIL.Image.open('jetpack.jpg')
display(Image('jetpack.jpg', width=300))

Now define a prompt to analyze the sample image:

In [ ]:
analyzePrompt = """
    This image contains a sketch of a potential product along with some notes.
    Given the product sketch, describe the product as thoroughly
    as possible based on what you see in the image, making sure to note
    all of the product features.

    Return output in json format.
"""

- Set the model to return JSON by setting `response_mime_type="application/json"`.
- Describe the schema for the response using a `TypedDict`.

In [ ]:
from typing_extensions import TypedDict

class Response(TypedDict):
  description: str
  features: list[str]

In [ ]:
from google.genai import types

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[analyzePrompt, img],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Response)
)

In [ ]:
import json

productInfo = json.loads(response.text)

print(json.dumps(productInfo, indent=4))

> Note: Here the model is just following text instructions for how the output json should be formatted. The API also supports a **strict JSON mode** where you specify a schema, and the API uses "Controlled Generation" (aka "Constrained Decoding") to ensure the model follows the schema, see the [JSON mode quickstart](https://github.com/google-gemini/cookbook/blob/main/quickstarts/JSON_mode.ipynb) for details.

## Generate marketing ideas

Now using the image you can use Gemini API to generate marketing names ideas:

In [ ]:
namePrompt = """
    You are a marketing whiz and writer trying to come up
    with a name for the product shown in the image.
    Come up with ten varied, interesting possible names.
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[namePrompt, img],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=list[str])
)

names = json.loads(response.text)
# Create a Dropdown widget to choose a name from the
# returned possible names
dropdown = widgets.Dropdown(
    options=names,
    value=names[0],  # default value
    description='Name:',
    disabled=False,
)
display(dropdown)

Finally you can work on generating a page for your product campaign:

In [ ]:
name = dropdown.value

In [ ]:
websiteCopyPrompt = f"""
  You're a marketing whiz and expert copywriter. You're writing
  website copy for a product named {name}. Your first job is to come
  up with H1 H2 copy. These are brief, pithy sentences or phrases that
  are the first and second things the customer sees when they land on the
  splash page. Here are some examples:
  [{{
    "h1": "A feeling is canned",
    "h2": "drinks and powders to help you feel calm cool and collected\
    despite the stressful world around you"
  }},
  {{
    "h1": "Design. Publish. Done.",
    "h2": "Stop rebuilding your designs from scratch. In Framer, everything\
    you put on the canvas is ready to be published to the web."
  }}]

  Create the same json output for a product named "{name}" with description\
  "{productInfo['description']}".
  Output ten different options as json in an array.
"""

In [ ]:
class Headings(TypedDict):
  h1:str
  h2:str

In [ ]:
copyResponse = client.models.generate_content(
    model=MODEL_ID,
    contents=[websiteCopyPrompt, img],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=list[Headings])
)

In [ ]:
copy = json.loads(copyResponse.text)

In [ ]:
print(json.dumps(copy, indent=4))

In [ ]:
h1 = copy[2]['h1']
h2 = copy[2]['h2']

In [ ]:
htmlPrompt = f"""
    Generate HTML and CSS for a splash page for a new product called {name}.
    Output only HTML and CSS and do not link to any external resources.
    Include the top level title: "{h1}" with the subtitle: "{h2}".

    Return the HTML directly, do not wrap it in triple-back-ticks (```).
"""

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[htmlPrompt])
print(response.text)

In [ ]:
HTML(response.text)